# Career & Skill Demand Intelligence Platform
## Data Preparation & Ingestion

### Tasks covered:
- Organize and load datasets (O\*NET Skills, Occupations, Education + BLS Wages)
- Explore datasets (shape, columns, dtypes, missing values)
- Identify and document important columns
- Clean column names and formats
- Standardize SOC Codes (remove `.00` from O\*NET)
- Extract preferred education level per occupation
- Merge all datasets into a unified career dataset
- Create data dictionary
- Save cleaned datasets (ready for GCS upload)

## Step 1 - Clone Repo & Import Libraries

In [ ]:
!git clone https://github.com/karan-cheemalapati/Career-Skill-Demand-Platform.git

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

## Step 2 — Load Datasets

In [ ]:
data_path = "/content/Career-Skill-Demand-Platform/data/raw_data/"

skills_df = pd.read_excel(data_path + "Skills.xlsx")
occupations_df = pd.read_excel(data_path + "Occupation_Data.xlsx")
education_df = pd.read_excel(data_path + "Education.xlsx")
wages_df = pd.read_excel(data_path + "national_M2024_dl.xlsx")

print("Datasets loaded")
print(f"Skills : {skills_df.shape}")
print(f"Occupations : {occupations_df.shape}")
print(f"Education : {education_df.shape}")
print(f"Wages (BLS) : {wages_df.shape}")

## Step 3 — Explore Datasets

In [ ]:
print("SKILLS DATASET")
display(skills_df.head())

print("\nOCCUPATIONS DATASET")
display(occupations_df.head())

print("\nEDUCATION DATASET")
display(education_df.head())

print("\nWAGES DATASET")
display(wages_df.head())

In [ ]:
for name, df in [('Skills', skills_df), ('Occupations', occupations_df), ('Education', education_df), ('Wages', wages_df)]:
  print(f"\n--- {name.upper()} INFO ---")
  df.info()

In [ ]:
for name, df in [('Skills', skills_df), ('Occupations', occupations_df), ('Education', education_df), ('Wages', wages_df)]:
  nulls = df.isnull().sum()
  nulls = nulls[nulls > 0]
  print(f"\nMissing values — {name}")
  if len(nulls):
    display(nulls)
  else:
    print("None")

## Step 4 — Clean Column Names

Uses regex to handle all special characters (`*`, `-`, spaces) so every column becomes clean `snake_case`.

In [ ]:
def clean_columns(df):
  """Lowercase + replace all non-alphanumeric chars with underscore + strip edges."""
  df.columns = (df.columns
                .str.strip()
                .str.lower()
                .str.replace(r'[^a-z0-9]+', '_', regex=True)
                .str.strip('_')
                )
  return df

skills_df = clean_columns(skills_df.copy())
occupations_df = clean_columns(occupations_df.copy())
education_df = clean_columns(education_df.copy())
wages_df = clean_columns(wages_df.copy())

print("Column names cleaned")
print("Skills :", list(skills_df.columns))
print("Occupations :", list(occupations_df.columns))
print("Education :", list(education_df.columns))
print("Wages :", list(wages_df.columns))

## Step 5 — Standardize SOC Codes (Remove `.00` from O\*NET)

Uses `str.replace(r'\.00$', '', regex=True)` — only removes the `.00` suffix, keeping specialty codes like `11-1011.03` intact.

In [ ]:
def standardize_soc(series):
  return series.astype(str).str.replace(r'\.00$', '', regex=True).str.strip()

skills_df['soc_code'] = standardize_soc(skills_df['o_net_soc_code'])
occupations_df['soc_code'] = standardize_soc(occupations_df['o_net_soc_code'])
education_df['soc_code'] = standardize_soc(education_df['o_net_soc_code'])
wages_df['soc_code'] = wages_df['occ_code'].str.strip()

print("SOC codes standardized")
print("\nBefore -> After sample (Skills):")
display(
    skills_df[['o_net_soc_code', 'soc_code']]
    .drop_duplicates()
    .head(8)
    .reset_index(drop=True)
)

## Step 6 — Select Important Columns

In [ ]:
skills_clean = skills_df[[
    'soc_code', 'title', 'element_id', 'element_name',
    'scale_id', 'scale_name', 'data_value', 'date', 'domain_source'
]].copy()

occupations_clean = occupations_df[['soc_code', 'title', 'description']].copy()

education_clean = education_df[[
    'soc_code', 'title', 'element_name', 'scale_id',
    'category', 'data_value', 'recommend_suppress'
]].copy()

wages_clean = wages_df[[
    'soc_code', 'occ_title', 'o_group', 'tot_emp',
    'h_mean', 'a_mean', 'h_median', 'a_median'
]].copy()

print("Important columns selected")
print(f"Skills clean : {skills_clean.shape}")
print(f"Occupations clean : {occupations_clean.shape}")
print(f"Education clean : {education_clean.shape}")
print(f"Wages clean : {wages_clean.shape}")

## Step 7 — Fix Data Types & BLS Suppression Values

BLS wage columns are stored as `object` because suppressed values use `#`. These are converted to numeric (`#` → `NaN`).

In [ ]:
wage_cols = ['h_mean', 'a_mean', 'h_median', 'a_median']

for col in wage_cols:
  wages_clean[col] = pd.to_numeric(
      wages_clean[col].astype(str)
      .str.replace(',', '')
      .str.strip()
      .replace({'#': np.nan, '*': np.nan}),
      errors='coerce'
      )

wages_clean['tot_emp'] = pd.to_numeric(
    wages_clean['tot_emp'].astype(str).str.replace(',', '').str.strip(),
    errors='coerce'
)

print("Wage data types fixed")
print(wages_clean[wage_cols + ['tot_emp']].dtypes)

## Step 8 — Extract Preferred Education Level per Occupation

The Education dataset stores 12 education categories per occupation with a `data_value` (% of workers who have that level). We extract the **most preferred** education level by picking the category with the highest `data_value` per occupation.

### Education Category Mapping (O\*NET RL Scale)
| Category | Education Level |
|---|---|
| 1 | Less than a High School Diploma |
| 2 | High School Diploma or GED |
| 3 | Post-Secondary Certificate |
| 4 | Some College Courses |
| 5 | Associate's Degree |
| 6 | Bachelor's Degree |
| 7 | Post-Baccalaureate Certificate |
| 8 | Master's Degree |
| 9 | Post-Master's Certificate |
| 10 | First Professional Degree (e.g. JD, MD) |
| 11 | Doctoral Degree |
| 12 | Post-Doctoral Training |

In [ ]:
# Education level label mapping (O*NET RL scale categories 1-12)
EDUCATION_MAP = {
    1: "Less than High School",
    2: "High School Diploma or GED",
    3: "Post-Secondary Certificate",
    4: "Some College Courses",
    5: "Associate's Degree",
    6: "Bachelor's Degree",
    7: "Post-Baccalaureate Certificate",
    8: "Master's Degree",
    9: "Post-Master's Certificate",
    10: "First Professional Degree (e.g. JD, MD)",
    11: "Doctoral Degree",
    12: "Post-Doctoral Training"
}

# Filter to: Required Level of Education only, drop suppressed rows
edu_required = education_clean[
    (education_clean['element_name'] == 'Required Level of Education') &
    (education_clean['recommend_suppress'] != 'Y') &
    (education_clean['data_value'] > 0)
].copy()

# For each occupation, pick the category with the highest data_value
edu_preferred = (
    edu_required
    .sort_values('data_value', ascending=False)
    .drop_duplicates(subset='soc_code', keep='first')
    [['soc_code', 'category', 'data_value']]
    .rename(columns={
        'category'   : 'preferred_edu_category',
        'data_value' : 'preferred_edu_pct'
    })
    .copy()
)

# Map category number to readable label
edu_preferred['preferred_education'] = (
    edu_preferred['preferred_edu_category']
    .astype(int)
    .map(EDUCATION_MAP)
)

print(f"Education levels extracted: {edu_preferred.shape[0]} occupations")
print("\nSample:")
display(edu_preferred.head(10).reset_index(drop=True))

In [ ]:
# Distribution of preferred education levels across all occupations
edu_dist = edu_preferred['preferred_education'].value_counts().sort_values(ascending=True)

plt.figure(figsize=(10, 6))
edu_dist.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title("Preferred Education Level Distribution Across Occupations", fontsize=13)
plt.xlabel("Number of Occupations")
plt.tight_layout()
plt.show()

print("\nCounts:")
display(edu_dist.sort_values(ascending=False))

## Step 9 — Filter Skills & Merge All Datasets

In [ ]:
# Filter skills to Importance scale only (scale_id = 'IM', NOT scale_name)
skills_importance = skills_clean[skills_clean['scale_id'] == 'IM'].copy()

print(f"Skills filtered to Importance (IM) scale: {skills_importance.shape}")
display(skills_importance.head())

In [ ]:
# Step 1: Skills + Occupation descriptions
skills_merged = pd.merge(
    skills_importance,
    occupations_clean[['soc_code', 'description']],
    on='soc_code',
    how='left'
)

# Step 2: Add preferred education level
skills_merged = pd.merge(
    skills_merged,
    edu_preferred[['soc_code', 'preferred_education', 'preferred_edu_pct']],
    on='soc_code',
    how='left'
)

print(f"After adding education: {skills_merged.shape}")
print(f"Education match rate: {skills_merged['preferred_education'].notna().sum() / len(skills_merged) * 100:.1f}%")

In [ ]:
# Step 3: Add BLS wages (detailed-level only, match on 7-char SOC base)
wages_detail = wages_clean[wages_clean['o_group'] == 'detailed'].copy()
wages_detail['soc_base']  = wages_detail['soc_code'].str[:7]
skills_merged['soc_base'] = skills_merged['soc_code'].str[:7]

career_dataset = pd.merge(
    skills_merged,
    wages_detail[['soc_base', 'tot_emp', 'a_mean', 'a_median']],
    on='soc_base',
    how='left'
).drop(columns='soc_base')

print(f"\nFinal career dataset: {career_dataset.shape}")
print(f"  Wage match rate      : {career_dataset['a_mean'].notna().sum() / len(career_dataset) * 100:.1f}%")
print(f"  Education match rate : {career_dataset['preferred_education'].notna().sum() / len(career_dataset) * 100:.1f}%")

print("\nPreview:")
display(career_dataset[[
    'soc_code', 'title', 'element_name', 'data_value',
    'preferred_education', 'preferred_edu_pct', 'a_mean'
]].head(10))

## Step 10 — Data Dictionary

In [ ]:
data_dict = pd.DataFrame({
    'dataset': [
        'skills_clean','skills_clean','skills_clean','skills_clean','skills_clean','skills_clean',
        'occupations_clean','occupations_clean','occupations_clean',
        'education_clean','education_clean','education_clean','education_clean',
        'edu_preferred','edu_preferred','edu_preferred',
        'wages_clean','wages_clean','wages_clean','wages_clean','wages_clean'
    ],
    'column': [
        'soc_code','title','element_name','scale_id','scale_name','data_value',
        'soc_code','title','description',
        'soc_code','element_name','category','data_value',
        'soc_code','preferred_education','preferred_edu_pct',
        'soc_code','occ_title','tot_emp','a_mean','a_median'
    ],
    'type': [
        'string','string','string','string','string','float',
        'string','string','string',
        'string','string','float','float',
        'string','string','float',
        'string','string','float','float','float'
    ],
    'description': [
        'SOC code — .00 suffix removed. Primary join key.',
        'Occupation title from O*NET',
        'Skill name (e.g. Reading Comprehension, Active Listening)',
        'IM = Importance scale, LV = Level scale',
        'Full scale name (Importance / Level)',
        'Skill score on 1-5 scale',
        'SOC code. Primary join key.',
        'Occupation title',
        'Detailed O*NET occupation description',
        'SOC code. Primary join key.',
        'Education element name (e.g. Required Level of Education)',
        'Education category number (1=Less than HS to 12=Post-Doctoral)',
        'Percentage of workers with that education level',
        'SOC code. Primary join key.',
        'Most common required education level label (from category 1-12 mapping)',
        'Percentage of workers who hold the preferred education level',
        'BLS SOC code (e.g. 11-1011)',
        'BLS occupation title',
        'Total national employment estimate (BLS 2024)',
        'Mean annual wage USD — suppressed values converted to NaN',
        'Median annual wage USD — suppressed values converted to NaN'
    ],
    'source': [
        'O*NET','O*NET','O*NET','O*NET','O*NET','O*NET',
        'O*NET','O*NET','O*NET',
        'O*NET','O*NET','O*NET','O*NET',
        'O*NET (derived)','O*NET (derived)','O*NET (derived)',
        'BLS','BLS','BLS','BLS','BLS'
    ]
})

print("Data Dictionary")
display(data_dict)

## Step 11 — EDA & Visualizations

In [ ]:
# Top 10 highest paying occupations with their required education
top_salary = (
    career_dataset[['title', 'a_mean', 'preferred_education']]
    .drop_duplicates(subset='title')
    .dropna(subset=['a_mean'])
    .sort_values('a_mean', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("Top 10 Highest Paying Occupations with Preferred Education Level")
display(top_salary)

In [ ]:
# Average salary by preferred education level
salary_by_edu = (
    career_dataset[['title', 'preferred_education', 'a_mean']]
    .drop_duplicates(subset='title')
    .dropna()
    .groupby('preferred_education')['a_mean']
    .mean()
    .sort_values()
)

plt.figure(figsize=(11, 6))
salary_by_edu.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Average Annual Salary by Preferred Education Level', fontsize=13)
plt.xlabel('Average Annual Salary (USD)')
plt.tight_layout()
plt.show()

print("\nAverage salaries:")
display(salary_by_edu.sort_values(ascending=False).reset_index())

In [ ]:
# Top 10 most common skills across all occupations
top_skills = career_dataset['element_name'].value_counts().head(10)

plt.figure(figsize=(10, 5))
top_skills.plot(kind='barh', color='teal', edgecolor='white')
plt.title('Top 10 Most Common Skills Across Occupations', fontsize=13)
plt.xlabel('Number of Occupations')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Annual salary distribution
salary_data = career_dataset[['title', 'a_mean']].drop_duplicates().dropna()

plt.figure(figsize=(10, 5))
salary_data['a_mean'].hist(bins=40, color='steelblue', edgecolor='white')
plt.title('Annual Salary Distribution Across Occupations', fontsize=13)
plt.xlabel('Mean Annual Salary (USD)')
plt.ylabel('Number of Occupations')
plt.tight_layout()
plt.show()

## Step 12 — Save Cleaned Datasets & Push to GitHub
All files saved to `data/cleaned_data/` .

In [ ]:
import os
import subprocess
import getpass

repo_path   = "/content/Career-Skill-Demand-Platform"
output_path = f"{repo_path}/data/cleaned_data/"
os.makedirs(output_path, exist_ok=True)

# Save all cleaned datasets
skills_clean.to_csv(output_path + "skills_clean.csv",           index=False)
occupations_clean.to_csv(output_path + "occupations_clean.csv", index=False)
education_clean.to_csv(output_path + "education_clean.csv",     index=False)
edu_preferred.to_csv(output_path + "education_preferred.csv",   index=False)
wages_clean.to_csv(output_path + "wages_clean.csv",              index=False)
career_dataset.to_csv(output_path + "merged_career_dataset.csv",     index=False)
data_dict.to_csv(output_path + "data_dictionary.csv",                index=False)

# Also save data dictionary to docs/
docs_path = f"{repo_path}/docs/"
os.makedirs(docs_path, exist_ok=True)
data_dict.to_csv(docs_path + "data_dictionary.csv", index=False)

print("Files saved")

# Push to GitHub
subprocess.run(["git", "-C", repo_path, "config", "user.email", "karan@project.com"])
subprocess.run(["git", "-C", repo_path, "config", "user.name",  "Karan Cheemalapati"])
subprocess.run(["git", "-C", repo_path, "add", "data/cleaned_data/"])
subprocess.run(["git", "-C", repo_path, "add", "docs/"])
subprocess.run(["git", "-C", repo_path, "commit", "-m", "Add cleaned datasets — Task 1: Data Preparation & Ingestion"])

# Securely enter token — never saved in notebook
token = getpass.getpass("Enter your GitHub token: ")
subprocess.run(["git", "-C", repo_path, "remote", "set-url", "origin",
                f"https://{token}@github.com/karan-cheemalapati/Career-Skill-Demand-Platform.git"])
subprocess.run(["git", "-C", repo_path, "push"])

print("Files pushed to GitHub")